In [1]:
# pip install openai-agents
# Make sure Ollama is running: ollama serve
# Pull the model first: ollama pull minimax-m2.7:cloud

import asyncio
from agents import Agent, Runner, OpenAIChatCompletionsModel, AsyncOpenAI

# ── Point the OpenAI-compatible client at your local Ollama server ────────────
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Ollama ignores this, but the client requires a value
)

# ── Wrap it in the Agents SDK model object ────────────────────────────────────
model = OpenAIChatCompletionsModel(
    model="minimax-m2.7:cloud",
    openai_client=ollama_client,
)

# ── Create the Lahore Guide agent ─────────────────────────────────────────────
lahore_guide = Agent(
    name="Lahore Guide",
    instructions=(
        "You are an enthusiastic and knowledgeable local guide for Lahore, Pakistan. "
        "You know the city's history, culture, food, architecture, and hidden gems inside out. "
        "Give vivid, practical, and personal recommendations. "
        "Mention specific neighbourhoods, streets, and timings where helpful."
    ),
    model=model,
)


async def main():
    # ── Turn 1 ────────────────────────────────────────────────────────────────
    print("=== Turn 1 ===")
    result1 = await Runner.run(
        lahore_guide,
        "What are the must-visit historical sites in Lahore?",
    )
    print(result1.final_output)

    # ── Turn 2: use to_input_list() to carry the conversation forward ─────────
    print("\n=== Turn 2 ===")
    result2 = await Runner.run(
        lahore_guide,
        "Which of those is best to visit at sunset?",
        input=result1.to_input_list(),  # ← appends prior turns automatically
    )
    print(result2.final_output)

    # ── Turn 3 ────────────────────────────────────────────────────────────────
    print("\n=== Turn 3 ===")
    result3 = await Runner.run(
        lahore_guide,
        "Any nearby street food spots I shouldn't miss?",
        input=result2.to_input_list(),  # ← chains from turn 2
    )
    print(result3.final_output)


asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [2]:
import asyncio
from agents import Agent, Runner, OpenAIChatCompletionsModel, AsyncOpenAI

# ── 1. Ollama client (OpenAI-compatible, pointed at local server) ──────────────
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Ollama ignores this value, but the client requires it
)

# ── 2. Wrap the Ollama client in the Agents SDK model object ──────────────────
model = OpenAIChatCompletionsModel(
    model="minimax-m2.7:cloud",
    openai_client=ollama_client,
)

# ── 3. Create the Lahore Guide agent ──────────────────────────────────────────
lahore_guide = Agent(
    name="Lahore Guide",
    instructions=(
        "You are an enthusiastic and knowledgeable local guide for Lahore, Pakistan. "
        "You know the city's history, culture, food, architecture, and hidden gems inside out. "
        "Give vivid, practical, and personal recommendations. "
        "Mention specific neighbourhoods, streets, and timings where helpful."
    ),
    model=model,
)


# ── 4. Multi-turn conversation ────────────────────────────────────────────────
async def main():

    # Turn 1 — fresh conversation, just pass a plain string
    print("=== Turn 1 ===")
    result1 = await Runner.run(
        lahore_guide,
        "What are the must-visit historical sites in Lahore?",
    )
    print(result1.final_output)

    # Turn 2 — continue from Turn 1
    # to_input_list() returns the full history so far (user + assistant messages)
    # We append our next question as a new user message
    print("\n=== Turn 2 ===")
    result2 = await Runner.run(
        lahore_guide,
        input=result1.to_input_list()
        + [{"role": "user", "content": "Which of those is best to visit at sunset?"}],
    )
    print(result2.final_output)

    # Turn 3 — continue from Turn 2
    print("\n=== Turn 3 ===")
    result3 = await Runner.run(
        lahore_guide,
        input=result2.to_input_list()
        + [
            {
                "role": "user",
                "content": "Any nearby street food spots I shouldn't miss?",
            }
        ],
    )
    print(result3.final_output)


asyncio.run(main())


RuntimeError: asyncio.run() cannot be called from a running event loop